In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)
logging.getLogger('urllib3').setLevel(logging.ERROR)

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

E0000 00:00:1774960940.003765      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774960940.048276      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774960940.428975      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774960940.429009      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774960940.429020      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774960940.429023      55 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
# ================================================================
# 1. IMPORTS
# ================================================================
import pandas as pd
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from openai import OpenAI

In [3]:
# ================================================================
# 2. LLM CLIENT (ONLY for extracting keywords from user free text)
# ================================================================
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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"
)

def extract_keywords_from_user_input(user_text):
    prompt = f"""You are a medical keyword extraction tool.

From the patient's description below, extract ONLY short medical keywords or phrases.

INCLUDE:
- Symptoms (e.g. fever, chest pain, dizziness, rash)
- Conditions (e.g. diabetes, high blood pressure)
- Risk factors (e.g. smoking, family history)
- Demographics (e.g. age, gender)
- Behaviours (e.g. heavy alcohol use)

EXCLUDE:
- Filler words and sentences
- Non-medical terms

FORMAT: Return ONLY keywords separated by ' | ' on a single line. Nothing else.

PATIENT TEXT: {user_text}"""

    completion = client.chat.completions.create(
        model="meta/llama-3.3-70b-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=512,
        stream=False
    )
    return completion.choices[0].message.content.strip()

In [4]:
# ================================================================
# 3. LOAD DATA
# ================================================================
df = pd.read_csv('/kaggle/input/datasets/karekezijoelnew01/dataset/data.csv')
print(f"Total rows loaded: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

Total rows loaded: 1163
Columns: ['disease_name', 'symptoms', 'risk_factors', 'specialist']


In [5]:
# ================================================================
# 4. FILL MISSING VALUES
# ================================================================
for col in ['disease_name', 'symptoms', 'risk_factors', 'specialist']:
    df[col] = df[col].fillna('').astype(str).str.strip()

print(f"Rows after fill: {len(df)}")

Rows after fill: 1163


In [6]:
# ================================================================
# 5. DISEASE LOOKUP TABLE
# ================================================================
disease_lookup = (
    df[['disease_name', 'symptoms', 'risk_factors', 'specialist']]
    .drop_duplicates(subset='disease_name')
    .set_index('disease_name')
)
print(f"Unique diseases in lookup: {len(disease_lookup)}")

Unique diseases in lookup: 1163


In [7]:
# ================================================================
# 6. BUILD VOCABULARY + WEIGHTED ENCODING
# ================================================================
clinicalbert = SentenceTransformer('emilyalsentzer/Bio_ClinicalBERT')

WEIGHT_SYMPTOMS = 1.0
WEIGHT_RISK_FACTORS = 0.5

all_keywords = set()
for col in ['symptoms', 'risk_factors']:
    for text in df[col]:
        for kw in text.split('|'):
            kw = kw.strip().lower()
            if kw:
                all_keywords.add(kw)

keyword_list = sorted(all_keywords)
keyword_to_idx = {k: i for i, k in enumerate(keyword_list)}
vocab_size = len(keyword_list)
print(f"Vocabulary size: {vocab_size}")

# Assign weight based on which column keyword appears in most
keyword_weight = {}
for kw in keyword_list:
    sym_count = sum(1 for text in df['symptoms'] if kw in [k.strip().lower() for k in text.split('|')])
    rf_count = sum(1 for text in df['risk_factors'] if kw in [k.strip().lower() for k in text.split('|')])
    keyword_weight[kw] = WEIGHT_SYMPTOMS if sym_count >= rf_count else WEIGHT_RISK_FACTORS

# Encode vocab with ClinicalBERT for fuzzy matching
print("Encoding vocabulary with ClinicalBERT...")
keyword_embeddings = clinicalbert.encode(keyword_list, show_progress_bar=True)


No sentence-transformers model found with name emilyalsentzer/Bio_ClinicalBERT. Creating a new one with mean pooling.


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Vocabulary size: 8540
Encoding vocabulary with ClinicalBERT...


Batches:   0%|          | 0/267 [00:00<?, ?it/s]

In [8]:
# ================================================================
# 7. SYMPTOM RARITY INDEX
# ================================================================
symptom_disease_count = {}
for kw in keyword_list:
    count = 0
    for text in df['symptoms']:
        if kw in [k.strip().lower() for k in text.split('|')]:
            count += 1
    symptom_disease_count[kw] = count

In [9]:
# ================================================================
# 8. ENCODING FUNCTION
# ================================================================
def text_to_multihot(keyword_string):
    vec = np.zeros(vocab_size, dtype=np.float32)
    if not keyword_string.strip():
        return vec
    keywords = [k.strip().lower() for k in keyword_string.split('|') if k.strip()]
    for kw in keywords:
        if kw in keyword_to_idx:
            idx = keyword_to_idx[kw]
            vec[idx] = keyword_weight.get(kw, 1.0)
        else:
            kw_emb = clinicalbert.encode([kw])
            sims = cosine_similarity(kw_emb, keyword_embeddings)[0]
            best_idx = np.argmax(sims)
            if sims[best_idx] > 0.7:
                vec[best_idx] = keyword_weight.get(keyword_list[best_idx], 1.0)
    return vec

# Encode training data
print("Encoding training data...")
X_list = []
for _, row in df.iterrows():
    vec = np.zeros(vocab_size, dtype=np.float32)
    for kw in row['symptoms'].split('|'):
        kw = kw.strip().lower()
        if kw and kw in keyword_to_idx:
            vec[keyword_to_idx[kw]] = WEIGHT_SYMPTOMS
    for kw in row['risk_factors'].split('|'):
        kw = kw.strip().lower()
        if kw and kw in keyword_to_idx:
            vec[keyword_to_idx[kw]] = WEIGHT_RISK_FACTORS
    X_list.append(vec)

X = np.vstack(X_list)
print(f"X shape: {X.shape}")

Encoding training data...
X shape: (1163, 8540)


In [10]:
# ================================================================
# 9. ENCODE LABELS
# ================================================================
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['disease_name'])
num_classes = len(label_encoder.classes_)
print(f"Classes: {num_classes}")

Classes: 1163


In [11]:
# ================================================================
# 10. BUILD MODEL
# ================================================================
model = Sequential([
    Input(shape=(vocab_size,)),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

I0000 00:00:1774961055.062389      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13215 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │     4,372,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1163)           │       150,027 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,690,315 (17.89 MB)

 Trainable params: 4,688,779 (17.89 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [12]:
# ================================================================
# 11. DATA AUGMENTATION + TRAIN
# ================================================================
def augment_data(X_orig, y_orig, augments_per_sample=30, min_keep=2):
    X_aug_list = [X_orig.copy()]
    y_aug_list = [y_orig.copy()]

    for i in range(len(X_orig)):
        vec = X_orig[i]
        on_indices = np.where(vec > 0)[0]
        if len(on_indices) < min_keep:
            continue
        for _ in range(augments_per_sample):
            n_keep = np.random.randint(min_keep, len(on_indices) + 1)
            selected = np.random.choice(on_indices, size=n_keep, replace=False)
            new_vec = np.zeros_like(vec)
            new_vec[selected] = vec[selected]
            X_aug_list.append(new_vec.reshape(1, -1))
            y_aug_list.append(np.array([y_orig[i]]))

    X_aug = np.vstack(X_aug_list)
    y_aug = np.concatenate(y_aug_list)
    return X_aug, y_aug

print("Augmenting with partial symptom subsets...")
X_aug, y_aug = augment_data(X, y, augments_per_sample=30, min_keep=2)
print(f"Original: {len(X)} samples -> Augmented: {len(X_aug)} samples")

shuffle_idx = np.random.permutation(len(X_aug))
X_aug = X_aug[shuffle_idx]
y_aug = y_aug[shuffle_idx]

history = model.fit(
    X_aug, y_aug,
    epochs=50,
    batch_size=64,
    verbose=1
)

Augmenting with partial symptom subsets...
Original: 1163 samples -> Augmented: 36053 samples
Epoch 1/50


I0000 00:00:1774961063.410909     173 service.cc:152] XLA service 0x7defcc010170 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774961063.410960     173 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1774961063.808190     173 cuda_dnn.cc:529] Loaded cuDNN version 91002


 38/564 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0096 - loss: 7.0728

I0000 00:00:1774961066.368775     173 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


564/564 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.3634 - loss: 4.5841
Epoch 2/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9275 - loss: 0.3670
Epoch 3/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9500 - loss: 0.2244
Epoch 4/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9574 - loss: 0.1787
Epoch 5/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9632 - loss: 0.1486
Epoch 6/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9665 - loss: 0.1348
Epoch 7/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9691 - loss: 0.1211
Epoch 8/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9725 - loss: 0.1032
Epoch 9/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9686 - loss: 0.1118
Epoch 10/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9728 - loss: 0.0997
Epoch 11/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9767 - loss: 0.0848
Epoch 12/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accurac

In [13]:
# ================================================================
# 12. HELPER FUNCTIONS
# ================================================================

def predict_disease(keywords_text, top_k=3):
    multihot = text_to_multihot(keywords_text).reshape(1, -1)
    on_bits = np.where(multihot[0] > 0)[0]
    matched = [keyword_list[i] for i in on_bits]
    print(f"  [DEBUG] Matched {len(matched)} keywords: {matched}")

    probs = model.predict(multihot, verbose=0)[0]
    top_idx = np.argsort(probs)[::-1][:top_k]

    results = []
    for idx in top_idx:
        disease = label_encoder.inverse_transform([idx])[0]
        confidence = probs[idx]
        results.append({'disease': disease, 'confidence': confidence})
    return results


def get_disease_keywords(disease_name):
    """Get ALL keywords (symptoms + risk factors) for a disease."""
    if disease_name not in disease_lookup.index:
        return set()
    symptoms = disease_lookup.loc[disease_name, 'symptoms']
    risk_factors = disease_lookup.loc[disease_name, 'risk_factors']
    all_kws = set()
    for text in [symptoms, risk_factors]:
        for kw in text.split('|'):
            if kw.strip():
                all_kws.add(kw.strip().lower())
    return all_kws


def get_specialist(disease_name):
    if disease_name not in disease_lookup.index:
        return "general practitioner"
    return disease_lookup.loc[disease_name, 'specialist']


def is_similar_symptom(symptom, asked_set):
    """Check if symptom is too similar to one already asked."""
    s = symptom.lower().strip()
    for asked in asked_set:
        a = asked.lower().strip()
        if s in a or a in s:
            return True
        if s.rstrip('s') == a.rstrip('s'):
            return True
        s_words = set(s.split())
        a_words = set(a.split())
        if len(s_words) > 0 and len(a_words) > 0:
            overlap = len(s_words & a_words) / min(len(s_words), len(a_words))
            if overlap >= 0.8:
                return True
    return False


def keyword_to_question(keyword):
    """Convert a keyword into a natural yes/no question."""
    keyword = keyword.strip().lower()

    gerund_map = {
        'exercising': 'exercise', 'running': 'run', 'swimming': 'swim',
        'working': 'work', 'living': 'live', 'eating': 'eat',
        'drinking': 'drink', 'using': 'use', 'taking': 'take',
        'standing': 'stand', 'sitting': 'sit', 'smoking': 'smoke',
        'walking': 'walk', 'lifting': 'lift', 'sleeping': 'sleep',
    }

    risk_phrases = [
        'family history', 'exposure', 'surgery', 'transplant', 'history',
        'obesity', 'pregnancy', 'overweight', 'weakened immune',
        'chronic', 'previous', 'prior',
    ]
    if any(rp in keyword for rp in risk_phrases):
        return f"Do you have {keyword}?"

    if keyword.startswith('age') or keyword.startswith('older') or keyword.startswith('young'):
        return f"Are you {keyword}?"
    if keyword in ('male', 'female') or keyword.startswith('born'):
        return f"Are you {keyword}?"

    first_word = keyword.split()[0]
    if first_word in gerund_map:
        base = gerund_map[first_word]
        rest = ' '.join(keyword.split()[1:])
        return f"Do you {base} {rest}".strip() + "?"

    return f"Do you experience {keyword}?"


def pick_best_question(top_preds, user_kws, asked_set):
    """Pick the most discriminative keyword across top predicted diseases."""
    candidates = []

    relevant_preds = [p for p in top_preds[:3] if p['confidence'] >= 0.10]
    if not relevant_preds:
        relevant_preds = [top_preds[0]]

    for pred in relevant_preds:
        disease = pred['disease']
        all_kws = get_disease_keywords(disease)

        for kw in all_kws:
            if kw in user_kws or is_similar_symptom(kw, asked_set):
                continue

            presence = sum(
                1 for p in relevant_preds
                if kw in get_disease_keywords(p['disease'])
            )
            discriminative_score = 1.0 / presence
            candidates.append((kw, discriminative_score))

    if not candidates:
        return None

    candidates.sort(key=lambda x: (-x[1], x[0]))
    return candidates[0][0]

In [14]:
# ================================================================
# 13. FULL CONSULTATION
# ================================================================
NUM_FOLLOWUP_QUESTIONS = 9

def run_consultation():
    print("=" * 60)
    print("  CURAMEDICA - MEDICAL SYMPTOM CHECKER")
    print("=" * 60)

    # --- Step 1: Get user symptoms ---
    user_text = input("\nDescribe your symptoms: ").strip()
    while not user_text:
        print("  Please describe your symptoms.")
        user_text = input("Describe your symptoms: ").strip()

    print("\nAnalyzing your symptoms...")
    keywords = extract_keywords_from_user_input(user_text)
    print(f"Identified: {keywords}")

    # --- Step 2: Initial top 3 prediction (shown to user) ---
    initial_predictions = predict_disease(keywords, top_k=3)

    print("\n" + "=" * 60)
    print("  INITIAL ASSESSMENT (Top 3)")
    print("=" * 60)
    for i, pred in enumerate(initial_predictions, 1):
        print(f"  {i}. {pred['disease']} ({pred['confidence']:.1%})")

    # --- Step 3: 9 follow-up questions ---
    user_kws = {k.strip().lower() for k in keywords.split('|') if k.strip()}
    asked_set = set()
    confirmed_kws = []
    current_keywords = keywords  # running keyword string for re-prediction

    print(f"\n  Follow-up questions. Answer Yes or No.")
    print("-" * 60)

    for q in range(1, NUM_FOLLOWUP_QUESTIONS + 1):
        # Re-predict behind the scenes to pick the next best question
        current_preds = predict_disease(current_keywords, top_k=3)

        best_kw = pick_best_question(current_preds, user_kws, asked_set)

        if best_kw is None:
            print(f"\n  No more distinguishing questions available.")
            break

        # Ask the question
        question = keyword_to_question(best_kw)
        print(f"\n  {q}. {question}")
        while True:
            answer = input("     Your answer (yes/no): ").strip().lower()
            if answer in ['yes', 'no', 'y', 'n']:
                break
            print("     Please type Yes or No.")

        asked_set.add(best_kw)

        if answer in ['yes', 'y']:
            confirmed_kws.append(best_kw)
            user_kws.add(best_kw)
            current_keywords = current_keywords + ' | ' + best_kw

    # --- Step 4: Final single prediction (shown to user) ---
    final_preds = predict_disease(current_keywords, top_k=1)
    final_disease = final_preds[0]
    specialist = get_specialist(final_disease['disease'])

    print("\n" + "=" * 60)
    print("  FINAL ASSESSMENT")
    print("=" * 60)
    print(f"  {final_disease['disease']} -- {final_disease['confidence']:.2%} confidence")
    print(f"\n  Recommended specialist: {specialist}")
    print("=" * 60)


run_consultation()

  CURAMEDICA - MEDICAL SYMPTOM CHECKER



Describe your symptoms:  I have whiteheads and blackheads on my face, small red bumps and pimples keep appearing on my forehead, chest, upper back and shoulders, and I sometimes get large painful lumps under my skin.



Analyzing your symptoms...
Identified: whiteheads | blackheads | red bumps | pimples | painful lumps
  [DEBUG] Matched 5 keywords: ['blackheads', 'painful lump', 'pimples', 'red bumps', 'whiteheads']

  INITIAL ASSESSMENT (Top 3)
  1. Acne (77.1%)
  2. Lipoma (13.5%)
  3. Cervicitis (8.0%)

  Follow-up questions. Answer Yes or No.
------------------------------------------------------------
  [DEBUG] Matched 5 keywords: ['blackheads', 'painful lump', 'pimples', 'red bumps', 'whiteheads']

  1. Do you experience a teenager?


     Your answer (yes/no):  yes


  [DEBUG] Matched 6 keywords: ['a teenager', 'blackheads', 'painful lump', 'pimples', 'red bumps', 'whiteheads']

  2. Do you experience breakouts on chest?


...

  [DEBUG] Matched 11 keywords: ['a teenager', 'blackheads', 'breakouts on chest', 'breakouts on forehead', 'breakouts on shoulders', 'breakouts on upper back', 'friction or pressure on your skin', 'painful lump', 'pimples', 'red bumps', 'whiteheads']

  8. Do you experience large painful lumps under the skin?


     Your answer (yes/no):  no


  [DEBUG] Matched 11 keywords: ['a teenager', 'blackheads', 'breakouts on chest', 'breakouts on forehead', 'breakouts on shoulders', 'breakouts on upper back', 'friction or pressure on your skin', 'painful lump', 'pimples', 'red bumps', 'whiteheads']

  9. Do you experience small red bumps on face?


     Your answer (yes/no):  yes


  [DEBUG] Matched 12 keywords: ['a teenager', 'blackheads', 'breakouts on chest', 'breakouts on forehead', 'breakouts on shoulders', 'breakouts on upper back', 'friction or pressure on your skin', 'painful lump', 'pimples', 'red bumps', 'small red bumps on face', 'whiteheads']

  FINAL ASSESSMENT
  Acne -- 100.00% confidence

  Recommended specialist: dermatologist


In [17]:
# ================================================================
# 14. SAVE MODEL (uncomment when ready)
# ================================================================
SAVE_DIR = '/kaggle/working//model'
import os
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(os.path.join(SAVE_DIR, 'disease_classifier_v2.keras'))

with open(os.path.join(SAVE_DIR, 'label_encoder_v2.pkl'), 'wb') as f:
    pickle.dump(label_encoder, f)

np.save(os.path.join(SAVE_DIR, 'keyword_embeddings_v2.npy'), keyword_embeddings)

with open(os.path.join(SAVE_DIR, 'vocab_v2.pkl'), 'wb') as f:
    pickle.dump({
        'keyword_list': keyword_list,
        'keyword_to_idx': keyword_to_idx,
        'keyword_weight': keyword_weight,
        'symptom_disease_count': symptom_disease_count,
    }, f)

with open(os.path.join(SAVE_DIR, 'disease_lookup_v2.pkl'), 'wb') as f:
    pickle.dump(disease_lookup, f)

print("All files saved.")

All files saved.


In [16]:
import shutil
shutil.rmtree('/kaggle/working/model', ignore_errors=True)